In [1]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import os
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, precision_recall_curve
kagglehub.login()

# Download latest version
path = kagglehub.competition_download('sch2-reg-2026-d5-1')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/sch2-reg-2026-d5-1


In [2]:
p = os.path.join(path, 'train.csv')
train = pd.read_csv(p)
train.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,0,42,technician,married,secondary,no,7,no,no,cellular,25,aug,117,3,-1,0,unknown,0
1,1,38,blue-collar,married,secondary,no,514,no,no,unknown,18,jun,185,1,-1,0,unknown,0
2,2,36,blue-collar,married,secondary,no,602,yes,no,unknown,14,may,111,2,-1,0,unknown,0
3,3,27,student,single,secondary,no,34,yes,no,unknown,28,may,10,2,-1,0,unknown,0
4,4,26,technician,married,secondary,no,889,yes,no,cellular,3,feb,902,1,-1,0,unknown,1


In [3]:
p = os.path.join(path, 'test.csv')
test = pd.read_csv(p)
test.head()

,id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,750000,32,blue-collar,married,secondary,no,1397,yes,no,unknown,21,may,224,1,-1,0,unknown
1,750001,44,management,married,tertiary,no,23,yes,no,cellular,3,apr,586,2,-1,0,unknown
2,750002,36,self-employed,married,primary,no,46,yes,yes,cellular,13,may,111,2,-1,0,unknown
3,750003,58,blue-collar,married,secondary,no,-1380,yes,yes,unknown,29,may,125,1,-1,0,unknown
4,750004,28,technician,single,secondary,no,1950,yes,no,cellular,22,jul,181,1,-1,0,unknown


In [4]:
p = os.path.join(path, 'sample_submission.csv')
samples = pd.read_csv(p)
samples

,id,y
0,750000,0.5
1,750001,0.5
2,750002,0.5
3,750003,0.5
4,750004,0.5
...,...,...
249995,999995,0.5
249996,999996,0.5
249997,999997,0.5
249998,999998,0.5


In [5]:
train["id"].duplicated().value_counts() #No duplicate ids.

id
False    750000
Name: count, dtype: int64

In [6]:
train.info() #Must convert all of these into float32 types for features.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 18 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   id         750000 non-null  int64 
 1   age        750000 non-null  int64 
 2   job        750000 non-null  object
 3   marital    750000 non-null  object
 4   education  750000 non-null  object
 5   default    750000 non-null  object
 6   balance    750000 non-null  int64 
 7   housing    750000 non-null  object
 8   loan       750000 non-null  object
 9   contact    750000 non-null  object
 10  day        750000 non-null  int64 
 11  month      750000 non-null  object
 12  duration   750000 non-null  int64 
 13  campaign   750000 non-null  int64 
 14  pdays      750000 non-null  int64 
 15  previous   750000 non-null  int64 
 16  poutcome   750000 non-null  object
 17  y          750000 non-null  int64 
dtypes: int64(9), object(9)
memory usage: 103.0+ MB


In [7]:
train.job.value_counts()

job
management       175541
blue-collar      170498
technician       138107
admin.            81492
services          64209
retired           35185
self-employed     19020
entrepreneur      17718
unemployed        17634
housemaid         15912
student           11767
unknown            2917
Name: count, dtype: int64

In [8]:
job_mapping = {'management': 0, 'blue-collar': 1, 'technician': 2, 'admin.': 3, 'services': 4, 'retired': 5, 'self-employed': 6,
               'entrepreneur': 7, 'unemployed': 8, 'housemaid': 9, 'student': 10, 'unknown': 11}

jobs = train.job.map(job_mapping).astype('float32')
jobs

0          2.0
1          1.0
2          1.0
3         10.0
4          2.0
          ... 
749995     4.0
749996     5.0
749997     1.0
749998     2.0
749999     2.0
Name: job, Length: 750000, dtype: float32

In [9]:
married_mapping = {'married': 0, 'single': 1, 'divorced': 2}

married = train.marital.map(married_mapping).astype('float32')
married.value_counts()

marital
0.0    480759
1.0    194834
2.0     74407
Name: count, dtype: int64

In [10]:
print(train.education.unique())
edu_mapping = {'primary': 0, 'secondary': 1, 'tertiary': 2, 'unknown': 3}
educations = train.education.map(edu_mapping).astype('float32')

['secondary' 'primary' 'tertiary' 'unknown']


In [11]:
binary_mapping = {'no': 0, 'yes': 1}
default = train.default.map(binary_mapping).astype('float32')
loan = train.loan.map(binary_mapping).astype('float32')

In [12]:
print(train.contact.unique())
contact_mapping = {'unknown': 0, 'cellular': 1, 'telephone': 2}
contacts = train.contact.map(contact_mapping).astype('float32')

['cellular' 'unknown' 'telephone']


In [13]:
month_mapping = {'jan': 0, 'feb': 1, 'mar': 2, 'apr': 3, 'may': 4, 'jun': 5, 'jul': 6, 'aug': 7, 'sep': 8, 'oct': 9, 'nov': 10, 'dec': 11}
months = train.month.map(month_mapping).astype('float32')

In [14]:
train.poutcome.unique()
poutcome_mappings = {'unknown': 0, 'failure': 1, 'other': 2, 'success': 3}
poutcomes = train.poutcome.map(poutcome_mappings).astype('float32')

In [15]:
#Convert the others to float32s.
other_features = train.loc[:,['id','age','balance','day','duration','campaign','pdays','previous']].astype('float32')
other_features.head()

,id,age,balance,day,duration,campaign,pdays,previous
0,0.0,42.0,7.0,25.0,117.0,3.0,-1.0,0.0
1,1.0,38.0,514.0,18.0,185.0,1.0,-1.0,0.0
2,2.0,36.0,602.0,14.0,111.0,2.0,-1.0,0.0
3,3.0,27.0,34.0,28.0,10.0,2.0,-1.0,0.0
4,4.0,26.0,889.0,3.0,902.0,1.0,-1.0,0.0


In [16]:
features = pd.concat([other_features, married,educations,default,loan,contacts,months,poutcomes],axis=1) #combine along the columns.
features.head()

,id,age,balance,day,duration,campaign,pdays,previous,marital,education,default,loan,contact,month,poutcome
0,0.0,42.0,7.0,25.0,117.0,3.0,-1.0,0.0,0.0,1.0,0.0,0.0,1.0,7.0,0.0
1,1.0,38.0,514.0,18.0,185.0,1.0,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,5.0,0.0
2,2.0,36.0,602.0,14.0,111.0,2.0,-1.0,0.0,0.0,1.0,0.0,0.0,0.0,4.0,0.0
3,3.0,27.0,34.0,28.0,10.0,2.0,-1.0,0.0,1.0,1.0,0.0,0.0,0.0,4.0,0.0
4,4.0,26.0,889.0,3.0,902.0,1.0,-1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0


In [17]:
married_t = test.marital.map(married_mapping).astype('float32')
educations_t = test.education.map(edu_mapping).astype('float32')
default_t = test.default.map(binary_mapping).astype('float32')
loan_t = test.loan.map(binary_mapping).astype('float32')
contacts_t = test.contact.map(contact_mapping).astype('float32')
months_t = test.month.map(month_mapping).astype('float32')
poutcomes_t = test.poutcome.map(poutcome_mappings).astype('float32')

other_features_t = test.loc[:,['id','age','balance','day','duration','campaign','pdays','previous']].astype('float32')
other_features_t.head()

features_t = pd.concat([other_features_t, married_t, educations_t, default_t, loan_t, contacts_t, months_t, poutcomes_t],axis=1).astype('float32')

In [18]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [19]:
#convert to numpy arrays
from sklearn.model_selection import train_test_split
x_data = features.values.astype('float32') 
y_data = train.y.values.astype('float32')

x_train,x_val,y_train,y_val=train_test_split(x_data,y_data,test_size=0.2,random_state=42,stratify=y_data)

x_data.shape, y_data.shape #750,000 examples, with 15 features.

((750000, 15), (750000,))

In [20]:
#scale the data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(features_t) #also the test set

print(f"Shape of scaled training data: {x_train.shape}")
print(f"Shape of scaled validation data: {x_val.shape}")

Shape of scaled training data: (600000, 15)
Shape of scaled validation data: (150000, 15)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [21]:
#convert to tensors
x_train = torch.from_numpy(x_train)
y_train = torch.from_numpy(y_train)
x_val = torch.from_numpy(x_val)
y_val = torch.from_numpy(y_val)
x_test = torch.from_numpy(x_test)
#test data has no labels.

print(f"Shape of scaled training data: {x_train.shape}")

Shape of scaled training data: torch.Size([600000, 15])


In [22]:
class BinaryBankNeuralNetwork(torch.nn.Module):
  def __init__(self,input_size):
    super(BinaryBankNeuralNetwork,self).__init__()
    self.linear1 = torch.nn.Linear(input_size,7)
    self.linear2 = torch.nn.Linear(7,3)
    self.linear3 = torch.nn.Linear(3,1)
    self.relu = torch.nn.ReLU()

  def forward(self,x):
    return self.linear3(self.relu(self.linear2(self.relu(self.linear1(x)))))

In [23]:
classifier = BinaryBankNeuralNetwork(15).to(device)

In [24]:
loss_fn = torch.nn.BCEWithLogitsLoss() #takes logits as inputs.
optimizer = torch.optim.Adam(classifier.parameters(),lr=0.02)

In [25]:
classifier.state_dict()

OrderedDict([('linear1.weight',
              tensor([[-0.1273, -0.1488,  0.0623, -0.1798,  0.2102, -0.0585, -0.1470,  0.0012,
                       -0.1037,  0.1706, -0.1291,  0.2187,  0.0034,  0.0929,  0.1586],
                      [ 0.0618,  0.0731,  0.0783,  0.0777, -0.0055,  0.0219,  0.1459,  0.0684,
                        0.2122, -0.0741,  0.0845,  0.0316, -0.0368, -0.0697, -0.2469],
                      [-0.0815, -0.0733,  0.0719, -0.2090,  0.1178,  0.0436,  0.0268, -0.0073,
                       -0.2203,  0.2561, -0.0102, -0.2545, -0.0101, -0.1643,  0.0072],
                      [-0.1686, -0.0127,  0.0041,  0.0748,  0.1151, -0.0339,  0.0084, -0.1737,
                       -0.0715, -0.1575,  0.1788,  0.1974, -0.1018,  0.1604, -0.0695],
                      [ 0.2475,  0.2332, -0.0241, -0.1928, -0.0243,  0.2338,  0.0708, -0.0577,
                        0.1874, -0.1151, -0.0199,  0.2539, -0.2215, -0.0964,  0.0752],
                      [-0.2189,  0.0318,  0.2234, -0.0461,

In [26]:
from torch.utils.data import TensorDataset, DataLoader
batch=500

#shuffle data as the model is learning from training data.
train_dataset = TensorDataset(x_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=batch, shuffle=True)

#no need to shuffle, since the model is predicting, not learning.
val_dataset = TensorDataset(x_val,y_val)
val_dataloader = DataLoader(val_dataset, batch_size=batch, shuffle=False)

test_dataset = TensorDataset(x_test)
test_dataloader = DataLoader(test_dataset, batch_size=batch, shuffle=False)

print(f"Number of batches in train dataloader: {len(train_dataloader)}")
print(f"Number of batches in validation dataloader: {len(val_dataloader)}")
print(f"Number of batches in test dataloader: {len(test_dataloader)}")

Number of batches in train dataloader: 1200
Number of batches in validation dataloader: 300
Number of batches in test dataloader: 500


In [27]:
torch.sigmoid(classifier(x_train[:10].to(device)))
#Before training, the first 10 examples' probabilites in the training dataset.

tensor([[0.5723],
        [0.5723],
        [0.5931],
        [0.5668],
        [0.5808],
        [0.6224],
        [0.5614],
        [0.5796],
        [0.5602],
        [0.6100]], device='cuda:0', grad_fn=<SigmoidBackward0>)

In [28]:
loss_hist = []
epochs = 100

for epoch in range(epochs):
    loss_g = 0

    for xb,yb in train_dataloader: #for each batch
      outputs = classifier(xb.to(device))
      loss = loss_fn(outputs,yb.unsqueeze(1).to(device))

      loss.backward() #compute gradients (backpropagation)
      optimizer.step() #update loss w/ gradients, to minimize loss
      optimizer.zero_grad() #zero out gradients, to avoid accumulation of gradients in calculations
      loss_g += loss.item() #add loss to loss_g, which will be added after every epoch to loss_hist.

    if (epoch + 1) % 10 == 0:
      print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss_g/len(train_dataloader):.4f}')
      loss_hist.append(loss_g)
    loss_g=0

Epoch [10/100], Loss: 0.1878
Epoch [20/100], Loss: 0.1865
Epoch [30/100], Loss: 0.1862
Epoch [40/100], Loss: 0.1863
Epoch [50/100], Loss: 0.1862
Epoch [60/100], Loss: 0.1861
Epoch [70/100], Loss: 0.1860
Epoch [80/100], Loss: 0.1861
Epoch [90/100], Loss: 0.1863
Epoch [100/100], Loss: 0.1861


In [29]:
classifier.eval() #now evaluating.
preds = []
probs = []
targets= []

with torch.no_grad():
  for xb, yb in val_dataloader:
    probability = torch.sigmoid(classifier(xb.to(device))) #logit -> probability via sigmoid transformation.
    prediction = torch.round(probability) #rounds probabilities to either 0 (active) or 1 (churned).

    #record results to lists
    probs.extend(probability.cpu().numpy())
    preds.extend(prediction.cpu().numpy())
    targets.extend(yb.cpu().numpy())

print(classification_report(preds,targets,target_names = ['0','1']))

              precision    recall  f1-score   support

           0       0.97      0.93      0.95    137957
           1       0.48      0.72      0.58     12043

    accuracy                           0.92    150000
   macro avg       0.73      0.83      0.77    150000
weighted avg       0.94      0.92      0.92    150000



In [30]:
weights = classifier.linear1.weight.cpu().detach().numpy() #detach and move to CPU for numpy compatibility
print(weights.shape,'\n') #(4 output neurons, features)

feature_names = features.columns
feature_importance = np.mean(np.abs(weights), axis=0) #calculate the mean absolute weight for each feature

importance_df = pd.DataFrame({'Feature': feature_names,
                              'Importance': feature_importance}).sort_values(by='Importance',ascending=False)

print(importance_df)

(7, 15) 

      Feature  Importance
4    duration    6.085021
6       pdays    4.184095
13      month    2.050664
14   poutcome    1.961761
12    contact    1.821661
8     marital    1.581161
2     balance    1.540439
10    default    1.126542
1         age    0.756787
11       loan    0.666533
5    campaign    0.640796
7    previous    0.551278
3         day    0.528415
9   education    0.357819
0          id    0.061831
